# Diagnóstico de variables geográficas

Este cuaderno documenta el diagnóstico de las variables `CODIGO`, `DISTRITO`, `DEPARTAMENTO`, `MUNICIPIO` y `DEPARTAMENTAL`.

El objetivo es identificar problemas de calidad y proponer reglas para el plan de limpieza. **En esta etapa no se modifica ningún valor del conjunto de datos.** Todos los conteos se calculan directamente desde el archivo unificado para que el análisis permanezca sincronizado si cambian los insumos.

## 1. Preparación reproducible

El módulo `src/diagnostico_vianka.py` concentra las funciones del análisis. El notebook únicamente las ejecuta y presenta sus resultados. Esto evita duplicar lógica entre código y documentación.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.diagnostico_vianka import (
    cargar_datos,
    diagnostico_codigo,
    diagnostico_departamento,
    diagnostico_departamental,
    diagnostico_distrito,
    diagnostico_municipio,
    generar_reportes,
    problemas_vianka,
    resumen_variables,
)

df = cargar_datos()
print(f"Registros analizados: {len(df):,}")
print(f"Variables del archivo unificado: {len(df.columns)}")

## 2. Resumen de las variables asignadas

La tabla siguiente muestra tipo leído, valores faltantes, valores únicos y problemas básicos de espacios. Los identificadores y códigos se mantienen como texto para no perder ceros ni alterar su estructura.

In [ ]:
resumen = resumen_variables(df)
display(resumen)

faltantes = resumen.set_index("variable")["faltantes"]
con_faltantes = faltantes[faltantes > 0]
texto_faltantes = (
    ", ".join(f"`{v}` ({int(n):,})" for v, n in con_faltantes.items())
    if not con_faltantes.empty
    else "ninguna variable"
)
display(Markdown(
    "**Lectura:** las variables con valores faltantes son "
    f"{texto_faltantes}. El diagnóstico no los completa automáticamente."
))

## 3. Diagnóstico de `CODIGO`

Se verifica que el código conserve el patrón esperado, sea único y mantenga coherencia con la combinación departamento–municipio. Aunque contiene dígitos, debe tratarse como texto porque es un identificador, no una magnitud numérica.

In [ ]:
tabla_codigo = diagnostico_codigo(df)
display(tabla_codigo)

vacios_codigo = int(tabla_codigo.loc[
    tabla_codigo["revision"].eq("valores vacios"), "cantidad"
].iloc[0])
duplicados_codigo = int(tabla_codigo.loc[
    tabla_codigo["revision"].eq("codigos duplicados"), "cantidad"
].iloc[0])

display(Markdown(
    "**Conclusión:** `CODIGO` no requiere una corrección de contenido "
    f"si los vacíos ({vacios_codigo}) y duplicados ({duplicados_codigo}) "
    "permanecen en cero. La regla propuesta es conservarlo como texto y "
    "validar su formato en la etapa final."
))

## 4. Diagnóstico de `DISTRITO`

La variable combina valores vacíos, códigos incompletos y dos estructuras completas. La existencia de dos formatos no se interpreta automáticamente como error: primero debe contrastarse con el catálogo del MINEDUC.

In [ ]:
tabla_distrito = diagnostico_distrito(df)
display(tabla_distrito)

conteos_distrito = tabla_distrito.set_index("categoria")["cantidad"]
vacios_distrito = int(conteos_distrito.get("vacio", 0))
incompletos_distrito = int(conteos_distrito.get("incompleto", 0))
formatos_completos = [
    categoria
    for categoria in tabla_distrito["categoria"]
    if categoria.startswith("formato ")
]

display(Markdown(
    "**Conclusión:** se identificaron "
    f"{vacios_distrito:,} valores vacíos y {incompletos_distrito:,} "
    "códigos incompletos. Los formatos completos observados son "
    f"{', '.join(f'`{f}`' for f in formatos_completos)}. "
    "Los vacíos deben representarse como faltantes y los incompletos "
    "deben revisarse sin inventar dígitos."
))

## 5. Diagnóstico de `DEPARTAMENTO`

Se compara el dominio observado con el catálogo preliminar de departamentos. `CIUDAD CAPITAL` se analiza aparte porque proviene de una exportación específica del portal y no corresponde a un departamento oficial.

In [ ]:
tabla_departamento = diagnostico_departamento(df)
display(tabla_departamento)

conteos_departamento = tabla_departamento.set_index("revision")["cantidad"]
registros_capital = int(conteos_departamento.get("registros en CIUDAD CAPITAL", 0))
registros_sin_tilde = int(conteos_departamento.get("departamentos sin tilde normativa", 0))
fuera_catalogo = int(conteos_departamento.get("valores fuera del catalogo preliminar", 0))

display(Markdown(
    "**Conclusión:** la categoría especial `CIUDAD CAPITAL` afecta "
    f"{registros_capital:,} registros y debe mapearse geográficamente a "
    "Guatemala sin perder la información de capital. También se detectaron "
    f"{registros_sin_tilde:,} registros con nombres departamentales sin la "
    f"tilde normativa. Los valores fuera del catálogo preliminar son {fuera_catalogo:,}."
))

## 6. Diagnóstico de `MUNICIPIO`

El análisis cuenta pares únicos departamento–municipio mediante subconjuntos explícitos del DataFrame. Esto evita expresiones ambiguas y deja claro qué filas se excluyen al separar `CIUDAD CAPITAL`.

Las zonas capitalinas se preservan como información de sububicación; no deben descartarse al normalizar el municipio geográfico.

In [ ]:
tabla_municipio = diagnostico_municipio(df)
display(tabla_municipio)

conteos_municipio = tabla_municipio.set_index("revision")["cantidad"]
pares_total = int(conteos_municipio.get("pares departamento-municipio", 0))
pares_no_capital = int(conteos_municipio.get("pares fuera de CIUDAD CAPITAL", 0))
zonas_capitalinas = int(conteos_municipio.get("zonas de Ciudad Capital", 0))
homonimos = int(conteos_municipio.get("nombres repetidos en distintos departamentos", 0))

display(Markdown(
    "**Conclusión:** el conjunto contiene "
    f"{pares_total:,} pares departamento–municipio observados; "
    f"{pares_no_capital:,} quedan al excluir la categoría especial de capital. "
    f"Se identificaron {zonas_capitalinas:,} etiquetas de zona y "
    f"{homonimos:,} nombres municipales presentes en más de un departamento. "
    "Estos últimos no se consideran errores automáticamente, porque pueden ser homónimos válidos."
))

## 7. Diagnóstico de `DEPARTAMENTAL`

`DEPARTAMENTAL` representa una división administrativa educativa y no debe forzarse a ser idéntica a `DEPARTAMENTO`. Se distinguen coincidencias exactas, diferencias únicamente ortográficas y subdivisiones administrativas válidas.

In [ ]:
tabla_departamental = diagnostico_departamental(df)
display(tabla_departamental)

resumen_relaciones = ", ".join(
    f"{fila.relacion}: {int(fila.cantidad):,}"
    for fila in tabla_departamental.itertuples(index=False)
)
display(Markdown(
    "**Conclusión:** las relaciones observadas son "
    f"{resumen_relaciones}. La limpieza debe corregir únicamente diferencias "
    "ortográficas comprobadas y conservar las subdivisiones administrativas."
))

## 8. Problemas que alimentan el plan de limpieza

La tabla consolida los problemas cuantificados y la estrategia preliminar. Los números se generan en cada ejecución; el texto del problema no contiene cantidades fijas que puedan quedar desactualizadas.

In [ ]:
problemas = problemas_vianka(df)
display(problemas)

generar_reportes(df)
print("Reportes reproducibles actualizados en reports/diagnostico_inicial/")

## 9. Conclusión general

- `CODIGO` debe conservarse como texto y validarse, no transformarse como número.
- `DISTRITO` concentra valores faltantes, códigos incompletos y más de una estructura válida que requiere contraste con el catálogo.
- `DEPARTAMENTO` necesita normalización ortográfica y un tratamiento explícito para `CIUDAD CAPITAL`.
- `MUNICIPIO` debe validarse junto con `DEPARTAMENTO`; las zonas capitalinas y los homónimos requieren reglas específicas.
- `DEPARTAMENTAL` es una dimensión administrativa independiente, por lo que no debe igualarse automáticamente a `DEPARTAMENTO`.

Las reglas definitivas deben aprobarse en `docs/plan_limpieza.md` antes de modificar el conjunto de datos.